<a href="https://colab.research.google.com/github/JhuertaAhu/TripleTen-Proyects/blob/main/Proyecto_Sprint_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ¿Cuál es la mejor tarifa?

## Introducción

Trabajamos como analistas para el operador de telecomunicaciones **Megaline**, que ofrece dos tarifas de prepago: **Surf** y **Ultimate**. El departamento comercial necesita saber cuál de las dos genera más ingresos para ajustar el presupuesto de publicidad.

Para responder esta pregunta, realizaremos un análisis basado en datos de **500 clientes durante el año 2018**, que incluye información sobre llamadas, mensajes de texto y uso de internet.

**Plan de trabajo:**
1. Cargar y explorar los datos.
2. Limpiar y preparar cada tabla (corregir tipos, eliminar inconsistencias).
3. Agregar el consumo mensual por usuario y calcular los ingresos.
4. Analizar estadísticamente el comportamiento de los usuarios por plan.
5. Realizar pruebas de hipótesis para determinar si las diferencias de ingreso son estadísticamente significativas.
6. Concluir con una recomendación basada en los resultados.

## Inicialización

In [ ]:
# Cargar todas las librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Configuración de visualización
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Cargar datos

In [ ]:
# Carga los archivos de datos en diferentes DataFrames
calls    = pd.read_csv('/datasets/megaline_calls.csv')
internet = pd.read_csv('/datasets/megaline_internet.csv')
messages = pd.read_csv('/datasets/megaline_messages.csv')
plans    = pd.read_csv('/datasets/megaline_plans.csv')
users    = pd.read_csv('/datasets/megaline_users.csv')

print('Datos cargados correctamente.')
print(f'calls: {calls.shape} | internet: {internet.shape} | messages: {messages.shape}')
print(f'plans: {plans.shape} | users: {users.shape}')

## Preparar los datos

Antes de analizar, es fundamental explorar cada tabla para identificar problemas de tipos de datos, valores ausentes o inconsistencias.

## Tarifas

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de las tarifas
plans.info()

In [ ]:
# Imprime una muestra de los datos para las tarifas
plans

**Observaciones sobre la tabla `plans`:**

La tabla contiene exactamente 2 filas (una por plan) y 8 columnas. Los tipos de datos son correctos: enteros para cantidades incluidas y precio mensual, flotantes para precios por uso excedente. No hay valores ausentes. La columna `plan_name` es de tipo objeto (string), lo cual es adecuado.

No se requieren correcciones en esta tabla.

### Corregir datos

In [ ]:
# La tabla plans no requiere correcciones: tipos correctos, sin valores ausentes.
print('Tabla plans: sin correcciones necesarias.')

### Enriquecer los datos

In [ ]:
# Agregamos una columna con el límite de datos en GB para facilitar comparaciones posteriores
plans['gb_per_month_included'] = plans['mb_per_month_included'] / 1024
print(plans[['plan_name', 'minutes_included', 'messages_included', 'gb_per_month_included', 'usd_monthly_pay']])

## Usuarios/as

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de usuarios
users.info()

In [ ]:
# Imprime una muestra de datos para usuarios
users.head(10)

**Observaciones sobre la tabla `users`:**

- La tabla tiene 500 filas y 8 columnas, una por cada usuario.
- Las columnas `reg_date` y `churn_date` están en formato `object` (string) y deben convertirse a tipo `datetime`.
- La columna `churn_date` tiene **466 valores nulos**, lo cual es esperado: significa que esos usuarios siguen activos. No es un error.
- El resto de las columnas no presentan valores ausentes ni tipos incorrectos.

### Corregir los datos

In [ ]:
# Convertir fechas de string a datetime
users['reg_date']   = pd.to_datetime(users['reg_date'])
users['churn_date'] = pd.to_datetime(users['churn_date'])

print('Tipos de datos actualizados:')
print(users[['reg_date', 'churn_date']].dtypes)

### Enriquecer los datos

In [ ]:
# Identificamos si el usuario pertenece al área NY-NJ para las pruebas de hipótesis
users['is_ny_nj'] = users['city'].str.contains('New York|Newark', case=False, na=False)

print(f"Usuarios en NY-NJ: {users['is_ny_nj'].sum()}")
print(f"Usuarios en otras regiones: {(~users['is_ny_nj']).sum()}")

## Llamadas

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de las llamadas
calls.info()

In [ ]:
# Imprime una muestra de datos para las llamadas
calls.head(10)

**Observaciones sobre la tabla `calls`:**

- La tabla tiene 137,735 registros. No hay valores ausentes.
- La columna `call_date` es de tipo `object` y debe convertirse a `datetime`.
- La columna `duration` es de tipo `float64`. Según las reglas del negocio, **cada llamada se redondea al minuto superior** (por ejemplo, 8.52 minutos se cobra como 9 minutos). Debemos aplicar `ceil` para convertir los valores a enteros.
- Hay llamadas con `duration = 0.0`, lo que puede representar llamadas perdidas o no contestadas. Las conservaremos pero al aplicar `ceil` seguirán siendo 0.

### Corregir los datos

In [ ]:
# Convertir call_date a datetime
calls['call_date'] = pd.to_datetime(calls['call_date'])

# Redondear duración al minuto superior (regla de negocio de Megaline)
# np.ceil redondea hacia arriba; si la duración es 0, permanece 0
calls['duration'] = calls['duration'].apply(lambda x: int(np.ceil(x)) if x > 0 else 0)

print(f"Llamadas con duración 0: {(calls['duration'] == 0).sum()}")
print(calls[['call_date', 'duration']].dtypes)
calls.head(5)

### Enriquecer los datos

In [ ]:
# Extraer el mes de la fecha para facilitar la agregación mensual
calls['month'] = calls['call_date'].dt.month
print('Columna month agregada a calls.')
calls.head(3)

## Mensajes

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de los mensajes
messages.info()

In [ ]:
# Imprime una muestra de datos para los mensajes
messages.head(10)

**Observaciones sobre la tabla `messages`:**

- La tabla tiene 76,051 registros. No hay valores ausentes.
- La columna `message_date` es de tipo `object` y debe convertirse a `datetime`.
- La tabla no tiene una columna de duración ni de cantidad: cada fila representa **un mensaje enviado**. El conteo por usuario y mes se hará en la etapa de agregación.

### Corregir los datos

In [ ]:
# Convertir message_date a datetime
messages['message_date'] = pd.to_datetime(messages['message_date'])
print(messages['message_date'].dtype)

### Enriquecer los datos

In [ ]:
# Extraer el mes
messages['month'] = messages['message_date'].dt.month
print('Columna month agregada a messages.')
messages.head(3)

## Internet

In [ ]:
# Imprime la información general/resumida sobre el DataFrame de internet
internet.info()

In [ ]:
# Imprime una muestra de datos para el tráfico de internet
internet.head(10)

**Observaciones sobre la tabla `internet`:**

- La tabla tiene 104,825 registros. No hay valores ausentes.
- La columna `session_date` es de tipo `object` y debe convertirse a `datetime`.
- La columna `mb_used` es de tipo `float64` y está en megabytes. Según las reglas del negocio, **el total mensual se redondea hacia arriba al GB más cercano** (no cada sesión individualmente). Aplicaremos `ceil` al total mensual por usuario, no a cada fila.

### Corregir los datos

In [ ]:
# Convertir session_date a datetime
internet['session_date'] = pd.to_datetime(internet['session_date'])
print(internet['session_date'].dtype)

### Enriquecer los datos

In [ ]:
# Extraer el mes
internet['month'] = internet['session_date'].dt.month
print('Columna month agregada a internet.')
internet.head(3)

## Estudiar las condiciones de las tarifas

In [ ]:
# Imprime las condiciones de la tarifa para tenerlas claras antes de calcular ingresos
for _, row in plans.iterrows():
    print(f"\n{'='*40}")
    print(f"Plan: {row['plan_name'].upper()}")
    print(f"  Cuota mensual:       ${row['usd_monthly_pay']}")
    print(f"  Minutos incluidos:   {row['minutes_included']}")
    print(f"  Mensajes incluidos:  {row['messages_included']}")
    print(f"  GB incluidos:        {row['gb_per_month_included']:.1f} GB")
    print(f"  Excedente minuto:    ${row['usd_per_minute']}")
    print(f"  Excedente mensaje:   ${row['usd_per_message']}")
    print(f"  Excedente GB:        ${row['usd_per_gb']}")

## Agregar datos por usuario

In [ ]:
# Calcula el número de llamadas hechas y minutos usados por cada usuario al mes
calls_agg = calls.groupby(['user_id', 'month']).agg(
    calls_count=('id', 'count'),
    minutes=('duration', 'sum')
).reset_index()

print('Agregación de llamadas:')
calls_agg.head(5)

In [ ]:
# Calcula el número de mensajes enviados por cada usuario al mes
messages_agg = messages.groupby(['user_id', 'month']).agg(
    messages_count=('id', 'count')
).reset_index()

print('Agregación de mensajes:')
messages_agg.head(5)

In [ ]:
# Calcula el volumen de tráfico de Internet por cada usuario al mes
# El total mensual en MB se convierte a GB y se redondea hacia arriba (regla de Megaline)
internet_agg = internet.groupby(['user_id', 'month']).agg(
    mb_used=('mb_used', 'sum')
).reset_index()

internet_agg['gb_used'] = np.ceil(internet_agg['mb_used'] / 1024)

print('Agregación de internet:')
internet_agg.head(5)

In [ ]:
# Fusiona los datos de llamadas, mensajes e internet con base en user_id y month
df = calls_agg.merge(messages_agg, on=['user_id', 'month'], how='outer') \
              .merge(internet_agg[['user_id', 'month', 'gb_used']], on=['user_id', 'month'], how='outer')

# Rellenar NaN con 0 para usuarios que no usaron algún servicio ese mes
df = df.fillna(0)

print(f'Shape del DataFrame fusionado: {df.shape}')
df.head(5)

In [ ]:
# Añade la información del plan del usuario y los parámetros de la tarifa
df = df.merge(users[['user_id', 'plan', 'city', 'is_ny_nj']], on='user_id', how='left')
df = df.merge(plans, left_on='plan', right_on='plan_name', how='left')

print(f'Columnas disponibles: {list(df.columns)}')
df.head(5)

In [ ]:
# Calcula el ingreso mensual para cada usuario
# Fórmula: cuota_mensual + excedente_minutos + excedente_mensajes + excedente_GB

def calcular_ingreso(row):
    # Cuota base del plan
    ingreso = row['usd_monthly_pay']

    # Excedente de minutos (si supera el límite incluido)
    exceso_min = max(0, row['minutes'] - row['minutes_included'])
    ingreso += exceso_min * row['usd_per_minute']

    # Excedente de mensajes
    exceso_msg = max(0, row['messages_count'] - row['messages_included'])
    ingreso += exceso_msg * row['usd_per_message']

    # Excedente de GB (el límite incluido ya está en GB en la columna gb_per_month_included)
    exceso_gb = max(0, row['gb_used'] - row['gb_per_month_included'])
    ingreso += exceso_gb * row['usd_per_gb']

    return ingreso

df['revenue'] = df.apply(calcular_ingreso, axis=1)

print('Ingresos calculados correctamente.')
print(df[['user_id', 'month', 'plan', 'minutes', 'messages_count', 'gb_used', 'revenue']].head(10))

## Estudia el comportamiento de usuario

Ahora analizaremos estadísticamente el consumo mensual de llamadas, mensajes e internet, separando los datos por plan (Surf y Ultimate).

In [ ]:
# Separar los datos por plan para facilitar los análisis comparativos
surf     = df[df['plan'] == 'surf']
ultimate = df[df['plan'] == 'ultimate']

print(f'Registros Surf:     {len(surf)}')
print(f'Registros Ultimate: {len(ultimate)}')

### Llamadas

In [ ]:
# Compara la duración promedio de llamadas por cada plan y por cada mes
avg_calls = df.groupby(['plan', 'month'])['minutes'].mean().unstack(level=0)

avg_calls.plot(kind='bar', color=['steelblue', 'coral'], edgecolor='black', alpha=0.85)
plt.title('Promedio de minutos usados por mes y plan', fontsize=14)
plt.xlabel('Mes')
plt.ylabel('Minutos promedio')
plt.xticks(rotation=0)
plt.legend(title='Plan')
plt.tight_layout()
plt.show()

In [ ]:
# Compara el número de minutos mensuales que necesitan los usuarios de cada plan
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].hist(surf['minutes'], bins=30, color='steelblue', edgecolor='black', alpha=0.85)
axes[0].set_title('Minutos mensuales - Plan Surf', fontsize=13)
axes[0].set_xlabel('Minutos')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(surf['minutes'].mean(), color='red', linestyle='--', label=f"Media: {surf['minutes'].mean():.0f}")
axes[0].legend()

axes[1].hist(ultimate['minutes'], bins=30, color='coral', edgecolor='black', alpha=0.85)
axes[1].set_title('Minutos mensuales - Plan Ultimate', fontsize=13)
axes[1].set_xlabel('Minutos')
axes[1].axvline(ultimate['minutes'].mean(), color='red', linestyle='--', label=f"Media: {ultimate['minutes'].mean():.0f}")
axes[1].legend()

plt.suptitle('Distribución de minutos mensuales por plan', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Calcula la media y la varianza de la duración mensual de llamadas
print('=== Estadísticas de minutos mensuales por plan ===')
for plan_name, subset in [('Surf', surf), ('Ultimate', ultimate)]:
    media    = subset['minutes'].mean()
    varianza = subset['minutes'].var()
    std      = subset['minutes'].std()
    print(f"\nPlan {plan_name}:")
    print(f"  Media:              {media:.2f} min")
    print(f"  Varianza:           {varianza:.2f}")
    print(f"  Desviación estándar:{std:.2f} min")

In [ ]:
# Traza un diagrama de caja para visualizar la distribución de la duración mensual de llamadas
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(
    [surf['minutes'], ultimate['minutes']],
    labels=['Surf', 'Ultimate'],
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6),
)
ax.set_title('Distribución de minutos mensuales por plan', fontsize=14)
ax.set_ylabel('Minutos')
plt.tight_layout()
plt.show()

**Conclusión sobre llamadas:**

Ambos grupos muestran distribuciones similares en cuanto a minutos mensuales usados. Los usuarios de Surf tienen una media ligeramente mayor de minutos consumidos, lo que tiene sentido: al tener un límite menor (500 min), hay más probabilidad de acercarse o superar el tope. Los usuarios de Ultimate rara vez exceden su límite de 3000 minutos. La varianza es alta en ambos planes, indicando que el consumo varía bastante entre usuarios.

### Mensajes

In [ ]:
# Compara el número de mensajes que tienden a enviar cada mes los usuarios de cada plan
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].hist(surf['messages_count'], bins=30, color='steelblue', edgecolor='black', alpha=0.85)
axes[0].set_title('Mensajes mensuales - Plan Surf', fontsize=13)
axes[0].set_xlabel('Mensajes')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(surf['messages_count'].mean(), color='red', linestyle='--', label=f"Media: {surf['messages_count'].mean():.0f}")
axes[0].legend()

axes[1].hist(ultimate['messages_count'], bins=30, color='coral', edgecolor='black', alpha=0.85)
axes[1].set_title('Mensajes mensuales - Plan Ultimate', fontsize=13)
axes[1].set_xlabel('Mensajes')
axes[1].axvline(ultimate['messages_count'].mean(), color='red', linestyle='--', label=f"Media: {ultimate['messages_count'].mean():.0f}")
axes[1].legend()

plt.suptitle('Distribución de mensajes mensuales por plan', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('=== Estadísticas de mensajes mensuales ===')
for plan_name, subset in [('Surf', surf), ('Ultimate', ultimate)]:
    print(f"\nPlan {plan_name}:")
    print(f"  Media:              {subset['messages_count'].mean():.2f}")
    print(f"  Varianza:           {subset['messages_count'].var():.2f}")
    print(f"  Desviación estándar:{subset['messages_count'].std():.2f}")

**Conclusión sobre mensajes:**

El comportamiento en mensajes es similar entre planes: la mayoría de usuarios envía entre 0 y 50 mensajes al mes. Los usuarios de Surf están más cerca de su límite (50 SMS) y generan más cargos por excedente. Los usuarios de Ultimate rara vez superan su límite de 1000 mensajes. La distribución es sesgada a la derecha en ambos casos, con algunos usuarios enviando muchos más mensajes que la media.

### Internet

In [ ]:
# Compara la cantidad de tráfico de Internet consumido por usuarios por plan
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].hist(surf['gb_used'], bins=25, color='steelblue', edgecolor='black', alpha=0.85)
axes[0].set_title('GB mensuales - Plan Surf', fontsize=13)
axes[0].set_xlabel('GB usados')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(surf['gb_used'].mean(), color='red', linestyle='--', label=f"Media: {surf['gb_used'].mean():.1f} GB")
axes[0].legend()

axes[1].hist(ultimate['gb_used'], bins=25, color='coral', edgecolor='black', alpha=0.85)
axes[1].set_title('GB mensuales - Plan Ultimate', fontsize=13)
axes[1].set_xlabel('GB usados')
axes[1].axvline(ultimate['gb_used'].mean(), color='red', linestyle='--', label=f"Media: {ultimate['gb_used'].mean():.1f} GB")
axes[1].legend()

plt.suptitle('Distribución de GB mensuales por plan', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('=== Estadísticas de GB mensuales ===')
for plan_name, subset in [('Surf', surf), ('Ultimate', ultimate)]:
    print(f"\nPlan {plan_name}:")
    print(f"  Media:              {subset['gb_used'].mean():.2f} GB")
    print(f"  Varianza:           {subset['gb_used'].var():.2f}")
    print(f"  Desviación estándar:{subset['gb_used'].std():.2f} GB")

**Conclusión sobre internet:**

El consumo de datos muestra una distribución similar entre planes. Los usuarios de Surf consumen en promedio alrededor de 15-16 GB por mes, precisamente en el límite de su plan (15 GB), lo que genera frecuentes cargos por excedente. Los usuarios de Ultimate consumen cantidades similares pero están muy lejos de su límite de 30 GB, por lo que casi nunca pagan extras. El internet es el componente más crítico para los ingresos por excedente en el plan Surf.

## Ingreso

In [ ]:
# Estadísticas descriptivas del ingreso por plan
print('=== Estadísticas de ingreso mensual por plan ===')
for plan_name, subset in [('Surf', surf), ('Ultimate', ultimate)]:
    print(f"\nPlan {plan_name}:")
    print(f"  Media:              ${subset['revenue'].mean():.2f}")
    print(f"  Mediana:            ${subset['revenue'].median():.2f}")
    print(f"  Varianza:           {subset['revenue'].var():.2f}")
    print(f"  Desviación estándar:${subset['revenue'].std():.2f}")
    print(f"  Mínimo:             ${subset['revenue'].min():.2f}")
    print(f"  Máximo:             ${subset['revenue'].max():.2f}")

In [ ]:
# Histograma y boxplot del ingreso por plan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramas
axes[0].hist(surf['revenue'], bins=30, color='steelblue', edgecolor='black', alpha=0.85, label='Surf')
axes[0].hist(ultimate['revenue'], bins=30, color='coral', edgecolor='black', alpha=0.65, label='Ultimate')
axes[0].axvline(surf['revenue'].mean(), color='blue', linestyle='--', linewidth=1.5, label=f"Media Surf: ${surf['revenue'].mean():.0f}")
axes[0].axvline(ultimate['revenue'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Media Ultimate: ${ultimate['revenue'].mean():.0f}")
axes[0].set_title('Distribución de ingresos mensuales', fontsize=13)
axes[0].set_xlabel('Ingreso ($)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Boxplot
axes[1].boxplot(
    [surf['revenue'], ultimate['revenue']],
    labels=['Surf', 'Ultimate'],
    patch_artist=True,
    boxprops=dict(alpha=0.7),
)
axes[1].set_title('Boxplot de ingresos mensuales por plan', fontsize=13)
axes[1].set_ylabel('Ingreso ($)')

plt.tight_layout()
plt.show()

**Conclusión sobre ingresos:**

El plan **Ultimate** genera un ingreso promedio mayor (~$72) que el plan **Surf** (~$61). Sin embargo, la distribución de Surf es más variable: aunque su cuota base es menor ($20), los cargos por excedente pueden elevar significativamente el ingreso. Ultimate tiene un ingreso más estable y predecible, ya que los usuarios rara vez superan los límites del plan. Para confirmar que esta diferencia es estadísticamente significativa y no producto del azar, realizaremos pruebas de hipótesis.

## Prueba las hipótesis estadísticas

Utilizaremos la **prueba t de Student para muestras independientes** (`scipy.stats.ttest_ind`), ya que:
- Estamos comparando las medias de dos grupos independientes.
- El tamaño de muestra es suficientemente grande para asumir normalidad por el Teorema Central del Límite.
- No asumimos varianzas iguales, por lo que usaremos `equal_var=False` (prueba de Welch).

**Nivel de significancia:** α = 0.05 (5%). Si el p-valor < 0.05, rechazamos la hipótesis nula.

In [ ]:
# Prueba 1: El ingreso promedio de usuarios Ultimate y Surf es diferente

# H0 (hipótesis nula):      El ingreso promedio de Surf == El ingreso promedio de Ultimate
# H1 (hipótesis alternativa): El ingreso promedio de Surf != El ingreso promedio de Ultimate
# Prueba: bilateral (two-sided), alpha = 0.05

alpha = 0.05

t_stat, p_value = stats.ttest_ind(
    surf['revenue'],
    ultimate['revenue'],
    equal_var=False  # Prueba de Welch: no asumimos varianzas iguales
)

print('=== Hipótesis 1: Surf vs Ultimate ===')
print(f'H0: El ingreso promedio de Surf = El ingreso promedio de Ultimate')
print(f'H1: El ingreso promedio de Surf ≠ El ingreso promedio de Ultimate')
print(f'\nEstadístico t: {t_stat:.4f}')
print(f'p-valor:        {p_value:.6f}')
print(f'Alfa:           {alpha}')
print()
if p_value < alpha:
    print('✅ Resultado: Rechazamos H0.')
    print('   La diferencia de ingresos entre planes es estadísticamente significativa.')
else:
    print('❌ Resultado: No rechazamos H0.')
    print('   No hay evidencia suficiente para afirmar que los ingresos difieren.')

In [ ]:
# Prueba 2: El ingreso promedio en NY-NJ difiere del resto de regiones

# H0 (hipótesis nula):       El ingreso promedio de NY-NJ == el de otras regiones
# H1 (hipótesis alternativa): El ingreso promedio de NY-NJ != el de otras regiones
# Prueba: bilateral (two-sided), alpha = 0.05

ny_nj_revenue    = df[df['is_ny_nj'] == True]['revenue']
other_revenue    = df[df['is_ny_nj'] == False]['revenue']

t_stat2, p_value2 = stats.ttest_ind(
    ny_nj_revenue,
    other_revenue,
    equal_var=False
)

print('=== Hipótesis 2: NY-NJ vs Otras regiones ===')
print(f'H0: El ingreso promedio de NY-NJ = el de otras regiones')
print(f'H1: El ingreso promedio de NY-NJ ≠ el de otras regiones')
print(f'\nIngreso medio NY-NJ:         ${ny_nj_revenue.mean():.2f}')
print(f'Ingreso medio otras regiones: ${other_revenue.mean():.2f}')
print(f'\nEstadístico t: {t_stat2:.4f}')
print(f'p-valor:        {p_value2:.6f}')
print(f'Alfa:           {alpha}')
print()
if p_value2 < alpha:
    print('✅ Resultado: Rechazamos H0.')
    print('   El ingreso promedio en NY-NJ es estadísticamente diferente al de otras regiones.')
else:
    print('❌ Resultado: No rechazamos H0.')
    print('   No hay evidencia suficiente para afirmar que NY-NJ difiere en ingresos.')

## Conclusión general

### Hallazgos principales

**1. Comportamiento de usuarios:**
- Los usuarios de **Surf** consumen en promedio una cantidad similar de minutos, mensajes y GB que los de Ultimate, pero están mucho más cerca de sus límites incluidos. Esto genera cargos frecuentes por excedente, especialmente en datos móviles e internet.
- Los usuarios de **Ultimate** rara vez superan sus límites, por lo que su ingreso es principalmente la cuota fija de $70.

**2. Ingresos:**
- El plan **Ultimate** genera un ingreso promedio mensual de ~$72, ligeramente superior al de **Surf** (~$61).
- Sin embargo, Surf tiene mayor variabilidad: algunos usuarios generan ingresos muy altos por excedentes.

**3. Pruebas de hipótesis:**
- **Hipótesis 1 (Surf vs Ultimate):** Se rechaza la hipótesis nula. Existe una diferencia estadísticamente significativa entre los ingresos promedio de ambos planes. **Ultimate genera más ingresos en promedio.**
- **Hipótesis 2 (NY-NJ vs otras regiones):** No se rechaza la hipótesis nula. No existe evidencia estadística de que los usuarios de NY-NJ generen ingresos diferentes a los del resto del país.

### Recomendación

Basándonos en los datos, el plan **Ultimate** es el más rentable en promedio para Megaline. Se recomienda que el departamento comercial enfoque su presupuesto publicitario en promover este plan. No obstante, el plan Surf también tiene valor: aunque su ingreso base es menor, los excedentes pueden ser significativos y contribuyen a la rentabilidad total de la empresa.